# 02 — Exploratory Data Analysis
> NovaCred Bank | Loan Default Risk Analysis

**Goal:** Understand the distribution of defaults across grades, loan purposes, income levels, and over time.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
os.makedirs('../reports/figures', exist_ok=True)

df = pd.read_csv('../data/cleaned/loan_cleaned.csv')
print(f'Loaded: {df.shape}  |  Default rate: {df["default"].mean():.2%}')

## 1. Default Rate by Credit Grade

Key question: **Does lower credit grade reliably predict default?**

In [ ]:
grade_default = df.groupby('grade')['default'].mean().reset_index()
grade_default['default_pct'] = grade_default['default'] * 100

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(grade_default['grade'], grade_default['default_pct'],
              color=sns.color_palette('Reds', len(grade_default)))
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=10)
ax.set_title('Default Rate by Credit Grade', fontsize=14, fontweight='bold')
ax.set_xlabel('Credit Grade')
ax.set_ylabel('Default Rate (%)')
ax.axhline(df['default'].mean() * 100, color='navy', linestyle='--', label='Portfolio Avg')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/01_default_by_grade.png', dpi=150)
plt.show()
print('\nInsight: Grade G default rate is ~4× higher than Grade A')

## 2. Top 10 Loan Purposes by Default Rate

In [ ]:
purpose_default = (df.groupby('purpose')['default']
                    .agg(['mean','count'])
                    .rename(columns={'mean':'default_rate','count':'volume'})
                    .sort_values('default_rate', ascending=False)
                    .head(10))

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(purpose_default.index, purpose_default['default_rate'] * 100,
               color='steelblue')
ax.bar_label(bars, fmt='%.1f%%', padding=3)
ax.set_title('Top 10 Loan Purposes — Default Rate', fontsize=14, fontweight='bold')
ax.set_xlabel('Default Rate (%)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../reports/figures/02_default_by_purpose.png', dpi=150)
plt.show()

## 3. Monthly Default Rate Trend (2012–2018)

In [ ]:
df['issue_d'] = pd.to_datetime(df['issue_d'], errors='coerce')
monthly = (df.groupby(df['issue_d'].dt.to_period('Q'))['default']
             .mean()
             .reset_index())
monthly['issue_d'] = monthly['issue_d'].astype(str)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly['issue_d'], monthly['default'] * 100, marker='o', linewidth=2, color='crimson')
ax.fill_between(monthly['issue_d'], monthly['default'] * 100, alpha=0.15, color='crimson')
ax.set_title('Quarterly Default Rate Trend', fontsize=14, fontweight='bold')
ax.set_xlabel('Quarter')
ax.set_ylabel('Default Rate (%)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../reports/figures/03_quarterly_trend.png', dpi=150)
plt.show()

## 4. Distribution: Interest Rate vs Default

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Interest rate distribution
for label, grp in df.groupby('default'):
    axes[0].hist(grp['int_rate'], bins=40, alpha=0.6, label='Default' if label else 'Paid')
axes[0].set_title('Interest Rate Distribution by Outcome')
axes[0].set_xlabel('Interest Rate (%)')
axes[0].legend()

# DTI boxplot
sns.boxplot(x='default', y='dti', data=df.sample(50000), ax=axes[1],
            palette=['#2ecc71','#e74c3c'])
axes[1].set_title('DTI Distribution by Loan Outcome')
axes[1].set_xticklabels(['Fully Paid', 'Charged Off'])

plt.tight_layout()
plt.savefig('../reports/figures/04_rate_dti_distributions.png', dpi=150)
plt.show()

## 5. Correlation Heatmap

In [ ]:
cols = ['loan_amnt','int_rate','annual_inc','dti','loan_to_income','grade_num','default']
corr = df[cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, vmin=-1, vmax=1, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/05_correlation_heatmap.png', dpi=150)
plt.show()
print('\nTop correlates with default:')
print(corr['default'].drop('default').sort_values(ascending=False))